In [1]:
import os
import shutil

# 0. Bulletproof Setup: Check for repo and enforce working directory
repo_path = "/kaggle/working/rl_sf"
if not os.path.exists(repo_path):
    print("--> Repository missing. Cloning now...")
    !git clone -b optimization https://github.com/flaviogeuforbio/rl-with-sf-for-mujoco {repo_path}

# Change directory explicitly to where the script lives
%cd {repo_path}

--> Repository missing. Cloning now...
Cloning into '/kaggle/working/rl_sf'...
remote: Enumerating objects: 3632, done.
remote: Total 3632 (delta 0), reused 0 (delta 0), pack-reused 3632 (from 3)
Receiving objects: 100% (3632/3632), 625.29 MiB | 36.54 MiB/s, done.
Resolving deltas: 100% (655/655), done.
Updating files: 100% (2397/2397), done.
/kaggle/working/rl_sf


In [2]:
import torch
print("torch:", torch.version)
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

torch: <module 'torch.version' from '/usr/local/lib/python3.12/dist-packages/torch/version.py'>
cuda available: True
device: Tesla T4


In [3]:
%ls

ActorCritic.py                        quick_test.py
args_plot_average_results_walker.txt  render_agent.py
artifacts/                            requirements.txt
diagnose_feature_scales.py            run_behavioral_diagnostics_mod.py
diagnose_psi.py                       run_behavioral_diagnostics.py
diagnose_rollout_dynamics.py          run_psi_diagnostic_mod.py
OLD_2_train_cheetah_walker.py         run_psi_diagnostic.py
OLD_train_cheetah_walker.py           train_cheetah_walker.py
plot_average_results.py               train_sf_ddpg.py
PlotResults.py                        transfer_vs_scratch_comparison.pdf
plot_rollout_timeseries.py            transfer_vs_scratch_comparison.png
__pycache__/                          utils.py
QuickPlot.py                          zero_shot_eval.py


In [4]:
!python -m pip install mujoco

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.7/232.7 kB 5.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 77.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 14.0 MB/s eta 0:00:00


In [5]:
import os
import shutil

# --- PHASE 1 SMOKE TEST: INITIAL RUN ---

STEPS_PER_PHASE = "2000"  
SAVE_FREQ = "1000"
RUN_STEPS_LIMIT = "3000"
GAMMA_VAL = "0.99"
LAMBDA_Q = "1.0" 
LAMBDA_VEC = "1.0" 
#RESUME_DIR = "/kaggle/input/datasets/adrianoarceri/checkpoint" # change the last name accoording to how you name the dataset, and the previous folder is your username
# Here I don't need the resume_dir!!
RUN_NAME_SEQ = f"Walker_transfer_gamma_{GAMMA_VAL.replace('.', '_')}_lq_{LAMBDA_Q.replace('.', '_')}_lvec_{LAMBDA_VEC.replace('.', '_')}_stepsxphase_{STEPS_PER_PHASE}"

kaggle_output_folder = "/kaggle/working/transfer_learning_long_run"
os.makedirs(kaggle_output_folder, exist_ok=True)

local_path_seq = f"/kaggle/working/rl_sf/artifacts/walker/{RUN_NAME_SEQ}"

seed=1 
print(f"\n--- EXECUTING SEED {seed} ---")

print("-> Running Sequential Training...")
!python -u train_cheetah_walker.py --steps_per_phase {STEPS_PER_PHASE} --save_freq {SAVE_FREQ} --baseline --run_name {RUN_NAME_SEQ} --gamma {GAMMA_VAL} --lambda_q {LAMBDA_Q} --lambda_vec {LAMBDA_VEC} --seed {seed} --run_steps_limit {RUN_STEPS_LIMIT}

if os.path.exists(local_path_seq):
    final_dest_seq = os.path.join(kaggle_output_folder, RUN_NAME_SEQ)
    shutil.copytree(local_path_seq, final_dest_seq, dirs_exist_ok=True)
    print(f"--> Sequential data saved in: {final_dest_seq}")

print("\n--> Zipping results for download...")
%cd /kaggle/working/
!zip -r transfer_learning_long_run.zip transfer_learning_long_run/
%cd /kaggle/working/rl_sf


--- EXECUTING SEED 1 ---
-> Running Sequential Training...
Training SF-DDPG...
--- Starting Phase 0 (Cheetah) ---
--> [SAVE] Checkpoint saved at Phase 0, Step 1000
--> [SAVE] Checkpoint saved at Phase 0, Step 2000
Chunk limit reached. Exiting safely without closing the phase.
Training DDPG...
--- Starting Task 1 (Cheetah Forward) ---
--> [SAVE] Checkpoint saved at Phase 0, Step 1000
--> [SAVE] Checkpoint saved at Phase 0, Step 2000
Chunk limit reached. Exiting safely without closing the phase.
--> Sequential data saved in: /kaggle/working/transfer_learning_long_run/Walker_transfer_gamma_0_99_lq_1_0_lvec_1_0_stepsxphase_4000

--> Zipping results for download...
/kaggle/working
  adding: transfer_learning_long_run/ (stored 0%)
  adding: transfer_learning_long_run/Walker_transfer_gamma_0_99_lq_1_0_lvec_1_0_stepsxphase_4000/ (stored 0%)
  adding: transfer_learning_long_run/Walker_transfer_gamma_0_99_lq_1_0_lvec_1_0_stepsxphase_4000/seed_1/ (stored 0%)
  adding: transfer_learning_long_run/